<a href="https://colab.research.google.com/github/ubsuny/PHY386/blob/Homework2026/2026/FinalProject/Final_01_SeestarStarColors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Project — Topic 1: Seestar Star-Color Classification (42 pts)

## Learning Outcomes
- Load and inspect a stacked FITS image taken with the Seestar S30 Pro.
- Detect stars and extract aperture photometry with `photutils`.
- Build a color index from the RGB channels stored by the Seestar.
- Cross-match detected stars against **Gaia DR3** via `astroquery.gaia` to obtain real labels.
- Train Random Forest and k-Nearest Neighbors classifiers on the Gaia-labeled subset.
- Interpret the result on a color–magnitude diagram in terms of the Hertzsprung–Russell relation.

## GitHub Workflow
This project uses the fork workflow (same as HW4–HW6). Your pull request:
`yourfork/yourname-final` → `ubsuny/PHY386:Homework2026`. Reviewer `@laserlab`, milestone `Final-2026`. See [`pull_request_template_Final.md`](pull_request_template_Final.md) for the PR checklist.

## Background: Stars, Color, and Temperature
Stars radiate approximately as blackbodies. Hotter stars are bluer, cooler stars are redder. The classical **B−V color index** (magnitude in the blue Johnson-B filter minus magnitude in the visual V filter) decreases with increasing effective temperature.

On the **Hertzsprung–Russell (HR) diagram** — absolute magnitude vs. color — most stars fall on the *main sequence*, a band running from hot-blue (upper left) to cool-red (lower right). Cluster age matters: a young open cluster like the **Pleiades (M45, ~100 Myr)** still contains hot main-sequence B-type stars; an old globular like **M5 (~13 Gyr)** has burned those off and is dominated by evolved red giants.

The Seestar S30 Pro records color images (R, G, B channels). We cannot recover true Johnson B−V from these channels, but the ratio G/R (or a similar combination) is a *proxy* that correlates with stellar temperature. In this project you will:
1. Measure that proxy for stars you detect in the image.
2. Use **Gaia DR3** as ground truth — Gaia provides real $B_P - R_P$ colors and, in many cases, classification of the star.
3. Train a classifier on the Gaia-labeled subset of your detected stars and see whether it generalizes to unlabeled detections.

This is exactly the HW5 workflow (Decision Tree, k-Nearest Neighbors, Random Forest on Hipparcos giants vs. dwarfs) applied to our own data.

In [ ]:
# Install extras not in requirements_HW2026.txt (Colab or fresh envs)
# %pip install astropy photutils astroquery scikit-learn --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from astropy.wcs import WCS
from photutils.detection import DAOStarFinder
from photutils.aperture import CircularAperture, aperture_photometry

plt.rcParams.update({
    'font.size': 11, 'axes.labelsize': 11, 'axes.titlesize': 11,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
})

## Worked Example: Load a Seestar FITS and Show It
Two FITS files are already committed under `data/`:

- `data/M45_Pleiades_20250321.fit` — Pleiades (young open cluster, hot main-sequence stars)
- `data/M5_Globular_20250328.fit` — M5 (old globular cluster, red-giant dominated)

Pick whichever is most interesting to you, or do both and compare.

In [ ]:
FITS_PATH = "data/M45_Pleiades_20250321.fit"

with fits.open(FITS_PATH) as hdul:
    hdul.info()
    data = hdul[0].data        # shape: (3, H, W) if RGB, else (H, W)
    header = hdul[0].header

print("Data shape:", data.shape, "dtype:", data.dtype)
print("OBJECT  :", header.get('OBJECT'))
print("DATE-OBS:", header.get('DATE-OBS'))
print("EXPTIME :", header.get('EXPTIME'), 's')
print("Has WCS :", 'CTYPE1' in header)

In [ ]:
def rgb_to_grayscale(cube: np.ndarray) -> np.ndarray:
    """Collapse a (3, H, W) RGB cube to a single luminance image.

    Uses the ITU-R BT.601 luminance weights; any reasonable weighting works
    for star detection since we do photometry on each channel separately.

    Parameters
    ----------
    cube : np.ndarray
        Either (3, H, W) RGB or (H, W) monochrome.

    Returns
    -------
    np.ndarray
        (H, W) luminance image.
    """
    if cube.ndim == 2:
        return cube.astype(float)
    r, g, b = cube[0], cube[1], cube[2]
    return 0.299 * r + 0.587 * g + 0.114 * b

luma = rgb_to_grayscale(data)

fig, ax = plt.subplots(figsize=(6, 6))
# Stretch for display; use percentiles to avoid bright-star saturation washing out the faint stars.
vmin, vmax = np.percentile(luma, [5, 99.5])
ax.imshow(luma, vmin=vmin, vmax=vmax, cmap='gray', origin='lower')
ax.set_title(f"{header.get('OBJECT','?')} — {header.get('DATE-OBS','?')[:10]}")
ax.set_xlabel('pixel x')
ax.set_ylabel('pixel y')
ax.tick_params(which='both', direction='in', top=True, right=True)
plt.tight_layout()
plt.show()

## Part 1 — Star Detection and Photometry (12 pts)

### Task 1.1 — Detect stars with DAOStarFinder (4 pts)
Use `photutils.detection.DAOStarFinder` on the luminance image. Measure the background with `sigma_clipped_stats` and set your detection threshold to 5σ above the background.

### Task 1.2 — Aperture photometry on each channel (4 pts)
Place a `CircularAperture` of radius ~4 pixels on each detected source. Run `aperture_photometry` separately on the R, G, and B channels. Assemble the results in a pandas DataFrame with columns `x, y, flux_r, flux_g, flux_b`.

### Task 1.3 — Compute a color index and an instrumental magnitude (4 pts)
- Color proxy: $C \equiv -2.5 \log_{10}(F_B / F_R)$. (More negative = bluer.)
- Instrumental magnitude: $m_{\rm inst} \equiv -2.5 \log_{10}(F_G)$.

Plot a histogram of $C$ and a histogram of $m_{\rm inst}$; comment on their shapes.

In [ ]:
# Part 1: your code here
#
# Starter skeleton:
# mean, median, std = sigma_clipped_stats(luma, sigma=3.0)
# finder = DAOStarFinder(threshold=5 * std, fwhm=3.0)
# sources = finder(luma - median)
# positions = np.transpose([sources['xcentroid'], sources['ycentroid']])
# apertures = CircularAperture(positions, r=4.)
# flux_r = aperture_photometry(data[0], apertures)['aperture_sum']
# flux_g = aperture_photometry(data[1], apertures)['aperture_sum']
# flux_b = aperture_photometry(data[2], apertures)['aperture_sum']
# ...

## Part 2 — Cross-Match with Gaia DR3 (10 pts)

### Task 2.1 — Convert pixel positions to sky coordinates (3 pts)
Use the WCS stored in the FITS header (`astropy.wcs.WCS(header)`) to convert each detection's (x, y) pixel position to (RA, Dec) in degrees. Verify by plotting a few positions over the image and checking they land on the expected stars.

*If your FITS lacks a valid WCS*, use `astrometry.net` or `astropy.coordinates.SkyCoord` by hand using the pointing in the header as a rough solution; document what you did.

### Task 2.2 — Cone-search Gaia DR3 (4 pts)
Using the image center and a generous radius (e.g. 0.5°), query Gaia DR3 via `astroquery.gaia`:
```python
from astroquery.gaia import Gaia
from astropy.coordinates import SkyCoord
import astropy.units as u
center = SkyCoord(ra_center, dec_center, unit='deg')
gaia_tab = Gaia.cone_search_async(center, radius=0.5 * u.deg).get_results()
```
Pull `source_id, ra, dec, phot_g_mean_mag, bp_rp, parallax, teff_gspphot` (or `teff_val` — whichever is available in DR3).

### Task 2.3 — Match detections to Gaia by angular distance (3 pts)
Use `SkyCoord.match_to_catalog_sky` to pair each detection with its nearest Gaia source. Accept matches closer than ~2–3 arcseconds. Drop the rest.

In [ ]:
# Part 2: your code here

## Part 3 — Supervised Classification (12 pts)

Assign each matched star a label. A common simple choice (binary): **"hot"** if Gaia `bp_rp` < 0.7, **"cool"** if `bp_rp` ≥ 0.7 — roughly separating spectral types A–F from G–M. Or bin into 3–5 classes if you want a multi-class problem.

### Task 3.1 — Features and split (3 pts)
Features: your measured color $C$ and instrumental magnitude $m_{\rm inst}$ (2D). Label: the Gaia-derived class. Split 80/20 train/test (`sklearn.model_selection.train_test_split`, `random_state=42`).

### Task 3.2 — Train three classifiers (5 pts)
Train **Decision Tree**, **k-Nearest Neighbors**, and **Random Forest** — the same trio from HW5. Standardize features with `StandardScaler` before k-Nearest Neighbors. Report accuracy and the confusion matrix on the test set for each.

### Task 3.3 — Feature importance (4 pts)
Inspect the Random Forest's `feature_importances_`. Which of $C$ and $m_{\rm inst}$ matters more, and does that match your physical intuition?

In [ ]:
# Part 3: your code here
#
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.preprocessing import StandardScaler
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import confusion_matrix, accuracy_score

## Part 4 — Physical Interpretation (8 pts)

### Task 4.1 — Color–magnitude diagram (4 pts)
Plot $m_{\rm inst}$ vs. $C$ with points colored by **predicted** class. Invert the magnitude axis so brighter is up. Overlay the decision boundary of your best classifier (use `DecisionBoundaryDisplay.from_estimator` from sklearn).

### Task 4.2 — Compare to Gaia ground truth (2 pts)
Make the same plot colored by **Gaia** class. Visually judge agreement.

### Task 4.3 — Physics discussion (2 pts)
In 5–8 sentences: does the Pleiades / M5 contrast show up in your data (if you did both)? How does the main-sequence morphology compare to the textbook Hertzsprung–Russell diagram? What limits your classifier — photometry noise, saturation of bright stars, atmospheric reddening, the crude RGB-to-color-index proxy?

In [ ]:
# Part 4: your code here

## Submission

### Commit messages
**Good** ✅: `Add aperture photometry and Gaia cross-match for Pleiades`

**Bad** ❌: `wip`, `update`, `stuff`

### Checklist
- [ ] Notebook in `2026/FinalProject/<yourname>/`
- [ ] All code cells execute top-to-bottom on a fresh kernel
- [ ] Type annotations and NumPy-style docstrings on every function you wrote
- [ ] All plots labeled with units, `figsize=(6, 4)` (or justified otherwise)
- [ ] Tasks sum to ~42 points
- [ ] PR: `yourfork/yourname-final` → `ubsuny/PHY386:Homework2026`
- [ ] Reviewer `@laserlab`, milestone `Final-2026`
- [ ] AI usage disclosed in the PR description

## Resources
- [`astropy.io.fits` documentation](https://docs.astropy.org/en/stable/io/fits/)
- [`photutils` quick start](https://photutils.readthedocs.io/en/stable/getting_started/)
- [`astroquery.gaia`](https://astroquery.readthedocs.io/en/latest/gaia/gaia.html)
- [Gaia DR3 data model — `gaia_source`](https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_main_source_catalogue/ssec_dm_gaia_source.html)
- PHY386 plotting-style reference: [`2026/handson/ScientificPlottingMatplotlib.ipynb`](../handson/ScientificPlottingMatplotlib.ipynb)